# CRAFT Phase 3: Deployment, Quantization, and Evaluation
This notebook covers the final phase of the pipeline:
1. Merging the trained LoRA adapter weights back into the base Phi-3-Mini model.
2. Converting the merged model to GGUF format using llama.cpp.
3. Quantizing the GGUF model to Q4_K_M.
4. Running full benchmark evaluations on GSM8K, StrategyQA, and MMLU.
5. Uploading the finalized model weights directly to Hugging Face.

In [ ]:
# 0. Setup Kaggle Environment & Clone Repository
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    git_token = user_secrets.get_secret("GITHUB_PAT")
    username = "VishalGastu30"
    !git clone https://{git_token}@github.com/{username}/CRAFT-SLM-Reasoning.git
except:
    print("Could not find GITHUB_PAT secret, cloning public repo instead...")
    !git clone https://github.com/VishalGastu30/CRAFT-SLM-Reasoning.git

%cd CRAFT-SLM-Reasoning
!git pull

In [ ]:
# 1. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy huggingface_hub

In [ ]:
# 1.5 Find and Copy Uploaded Checkpoints from Kaggle Datasets
import os
import shutil

print("Scanning /kaggle/input for uploaded checkpoints...")
rl_checkpoint = None
sft_checkpoint = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'checkpoint-250' in dirs or 'checkpoint-300' in dirs:
        match = 'checkpoint-250' if 'checkpoint-250' in dirs else 'checkpoint-300'
        rl_checkpoint = os.path.join(root, match)
    # Also look for a raw folder that might just contain the adapter files
    elif 'adapter_model.safetensors' in files and not rl_checkpoint:
        if 'rl' in root.lower() or '250' in root or '300' in root:
            rl_checkpoint = root
        elif 'sft' in root.lower() or 'phase1' in root.lower():
            sft_checkpoint = root

if rl_checkpoint:
    print(f"🎉 Found RL checkpoint path: {rl_checkpoint}")
    target_dir = "/kaggle/working/CRAFT-SLM-Reasoning/checkpoints/rl/final"
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    os.makedirs(os.path.dirname(target_dir), exist_ok=True)
    shutil.copytree(rl_checkpoint, target_dir)
    print(f"✅ Successfully copied RL checkpoint to {target_dir}!")
else:
    print("❌ Could not find RL checkpoint-250 or 300 inside /kaggle/input.")

if sft_checkpoint:
    print(f"🎉 Found SFT checkpoint path: {sft_checkpoint}")
    sft_target = "/kaggle/working/CRAFT-SLM-Reasoning/checkpoints/sft/final"
    if os.path.exists(sft_target):
        shutil.rmtree(sft_target)
    os.makedirs(os.path.dirname(sft_target), exist_ok=True)
    shutil.copytree(sft_checkpoint, sft_target)
    print(f"✅ Successfully copied SFT checkpoint to {sft_target}!")


In [ ]:
# 2. Verify files and SFT/RL checkpoints exist
import os
assert os.path.exists("checkpoints/rl/final"), "Error: checkpoints/rl/final not found. Run Phase 2 RL first!"

In [ ]:
# 3. Run quantization and merge
!python -m src.phase3_deploy.quantizer --base-model microsoft/Phi-3-mini-4k-instruct --lora-adapter checkpoints/rl/final --merged-output craft_output/merged

In [ ]:
# 4. Clone and Build llama.cpp for GGUF compilation
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && make -j

In [ ]:
# 5. Convert and quantize to Q4_K_M
!python llama.cpp/convert_hf_to_gguf.py craft_output/merged --outfile craft_output/craft_phi3_raw.gguf
!./llama.cpp/llama-quantize craft_output/craft_phi3_raw.gguf craft_output/craft_phi3_Q4_K_M.gguf Q4_K_M
print("Final model size:")
!ls -lh craft_output/craft_phi3_Q4_K_M.gguf

In [ ]:
# 6. Run benchmarks and evaluations
!python -m src.phase3_deploy.evaluator --model-type gguf --model-path craft_output/craft_phi3_Q4_K_M.gguf --dataset all --samples 50 --output-json craft_output/results.json

In [ ]:
# 7. Hugging Face Login and Upload (COMMENTED OUT FOR TESTING)
# Uncomment the code below ONLY AFTER you verify the benchmark scores in Step 6!
"""
from huggingface_hub import HfApi
import os

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    api = HfApi(token=hf_token)
    print("Hugging Face API authenticated using secret token.")
    api.create_repo(repo_id="Aurvion/CRAFT-Phi3-Mini", repo_type="model", private=False, exist_ok=True)
    api.upload_file(
        path_or_fileobj="craft_output/craft_phi3_Q4_K_M.gguf",
        path_in_repo="craft_phi3_Q4_K_M.gguf",
        repo_id="Aurvion/CRAFT-Phi3-Mini"
    )
    print("GGUF model successfully uploaded to Hugging Face!")
else:
    print("HF_TOKEN environment variable not set. Please upload the GGUF model manually or login using huggingface-cli.")
"""
print("HF Upload is paused. Review your benchmark scores above first!")